# Fine-Tuning the BGE-M3 Embedder for English to German Clinical Retrieval

The central difficulty of the retrieval stage is cross-lingual: the practitioner's question
arrives in English while the AWMF knowledge base is written in German. Section 4.8 showed that
monolingual medical embedders collapse on this task and that the multilingual BAAI/bge-m3 is the
only viable base. This notebook examines whether bge-m3 can be sharpened for English to German
clinical alignment through contrastive fine-tuning on the project's parallel data.

The same guarantees as the generator experiment apply. Training pairs are drawn only from the
held-out benchmark rows, the 715 minus the 200-question evaluation subset, and the evaluation
subset is never trained on. The embedder is trained and evaluated locally with
sentence-transformers, with no external judge in the training loop.

The training signal is the cleanest available. Every benchmark question exists in both English
and German, so each row is a verified English/German pair. Drawing those pairs together in
embedding space teaches the model that an English clinical query and its German counterpart
should occupy the same region, which is the property that retrieval over the German corpus
depends on.

## 1. Environment

The notebook targets a Colab GPU runtime. bge-m3 is a large model; a T4 handles a small batch
size, and an A100 allows a larger one.

In [ ]:
%%capture
!pip install -q -U sentence-transformers datasets accelerate faiss-cpu

## 2. Configuration

As in the generator notebook, the column names are configurable and printed by the first data
cell.

In [ ]:
CONFIG = {
    "BENCHMARK_CSV": "GP_Top10_Bilingual_Open_EndedQ.csv",  # full 715-row bilingual benchmark
    "EVAL200_CSV":   "eval_200_subset.csv",                 # 200 questions used in Chapter 4 (held out of training)

    "ID_COL":   "id",
    "Q_EN_COL": "question_en",
    "Q_DE_COL": "question_de",

    "BASE_MODEL":  "BAAI/bge-m3",
    "OUTPUT_DIR":  "bge-m3-clinical-ende",

    "BATCH":  16,     # 8 on a T4 if memory is tight
    "EPOCHS": 2,
    "LR":     2e-5,

    # RAGAS retrieval evaluation (optional, comparable to Chapter 4). Generation-free; only the judge uses an API.
    "AWMF_CHUNKS_CSV": "awmf_chunks.csv",              # the AWMF chunks (one text column)
    "CHUNK_TEXT_COL":  "page_content",
    "TOP_K":  10,
    "GROUND_TRUTH_CSV": None,                          # corpus-grounded references (question -> reference); required for this cell
    "GT_Q_COL":   "question_en",
    "GT_REF_COL": "reference",
    "JUDGE_MODEL":    "openai/gpt-4o-mini",
    "JUDGE_BASE_URL": "https://openrouter.ai/api/v1",
}

## 3. Parallel English/German pairs from the held-out split

The evaluation subset is separated first so it can serve as an unseen test set in Section 5.
Training pairs come only from the remaining held-out rows.

In [ ]:
import pandas as pd

bench = pd.read_csv(CONFIG["BENCHMARK_CSV"])
print("benchmark rows:", len(bench))
print("columns:", bench.columns.tolist())

id_col, qen, qde = CONFIG["ID_COL"], CONFIG["Q_EN_COL"], CONFIG["Q_DE_COL"]
eval200 = pd.read_csv(CONFIG["EVAL200_CSV"])

if id_col in bench.columns and id_col in eval200.columns:
    keys = set(eval200[id_col].astype(str))
    held    = bench[~bench[id_col].astype(str).isin(keys)].copy()
    eval_df = bench[ bench[id_col].astype(str).isin(keys)].copy()
else:
    keys = set(eval200[qen].astype(str).str.strip())
    mask = bench[qen].astype(str).str.strip().isin(keys)
    held, eval_df = bench[~mask].copy(), bench[mask].copy()

held    = held.dropna(subset=[qen, qde])
eval_df = eval_df.dropna(subset=[qen, qde])
print(f"held-out training rows: {len(held)}   |   evaluation rows (untouched): {len(eval_df)}")

## 4. Contrastive fine-tuning with in-batch negatives

MultipleNegativesRankingLoss treats each English question as an anchor and its German counterpart
as the single positive, with every other German question in the batch acting as an implicit
negative. Minimising this loss draws each English query towards its own German translation and
away from unrelated German text, which is the geometry a cross-lingual retriever requires; no
manually labelled negatives are needed. The learning rate is small and the run short, because
bge-m3 is already competent and the aim is a gentle adjustment rather than retraining from
scratch.

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

model = SentenceTransformer(CONFIG["BASE_MODEL"], device="cuda")

train_pairs = list(zip(held[qen].astype(str), held[qde].astype(str)))
train_examples = [InputExample(texts=[en, de]) for en, de in train_pairs]
loader = DataLoader(train_examples, shuffle=True, batch_size=CONFIG["BATCH"])
loss = losses.MultipleNegativesRankingLoss(model)

model.fit(
    train_objectives = [(loader, loss)],
    epochs = CONFIG["EPOCHS"],
    warmup_steps = max(1, int(0.1 * len(loader))),
    optimizer_params = {"lr": CONFIG["LR"]},
    use_amp = True,
    show_progress_bar = True,
)
model.save(CONFIG["OUTPUT_DIR"])
print("saved fine-tuned embedder to", CONFIG["OUTPUT_DIR"])

## 5. Cross-lingual retrieval evaluation (English query to German target)

The evaluation runs on the 200-question subset, which the embedder did not see during training.
For each English question the model must rank that question's German counterpart at the top of
the German candidates. This is a label-free proxy for corpus retrieval, measuring English to
German alignment directly through the parallel questions. Three standard metrics are reported for
the base model and the fine-tuned model under identical conditions: P@1, the German counterpart
is ranked first; R@5, it appears in the top five; and MRR, the mean reciprocal rank of the correct
counterpart.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

queries_en = eval_df[qen].astype(str).tolist()
corpus_de  = eval_df[qde].astype(str).tolist()   # candidate index; the correct answer for query i is item i

def xling_scores(m):
    qv = m.encode(queries_en, normalize_embeddings=True, batch_size=32)
    cv = m.encode(corpus_de,  normalize_embeddings=True, batch_size=32)
    sims = qv @ cv.T
    ranks = []
    for i, srow in enumerate(sims):
        order = np.argsort(-srow)             # descending similarity
        ranks.append(int(np.where(order == i)[0][0]))
    ranks = np.array(ranks)
    return {
        "P@1": round(float((ranks == 0).mean()), 3),
        "R@5": round(float((ranks < 5).mean()), 3),
        "MRR": round(float((1.0 / (ranks + 1)).mean()), 3),
    }

base_model  = SentenceTransformer(CONFIG["BASE_MODEL"], device="cuda")
tuned_model = SentenceTransformer(CONFIG["OUTPUT_DIR"], device="cuda")

results = pd.DataFrame([
    {"model": "bge-m3 (base)",       **xling_scores(base_model)},
    {"model": "bge-m3 (fine-tuned)", **xling_scores(tuned_model)},
])
results

## 6. RAGAS retrieval evaluation (comparable to Chapter 4, generation-free)

The P@1, R@5, and MRR values above use the parallel German questions as targets, which is an
alignment proxy. The metrics the chapter reports for embedders in Table 4.2 and Table 4.3 are
RAGAS Context Precision and Context Recall, measured over the retrieved AWMF chunks. Both depend
only on the question, the retrieved contexts, and a reference answer, with no answer generation
required, so this cell re-embeds the corpus with each embedder, retrieves, and scores retrieval
quality directly. The only external call is the judge.

The cell requires a corpus-grounded reference file, GROUND_TRUTH_CSV, mapping each question to its
reference answer as built in Section 4.14. A fully offline run is possible by setting
JUDGE_BASE_URL to a local OpenAI-compatible endpoint.

In [ ]:
!pip install -q ragas langchain-openai langchain-huggingface faiss-cpu

import os, numpy as np, faiss
os.environ.setdefault("OPENROUTER_API_KEY", "")  # judge API key (environment variable)

gt = pd.read_csv(CONFIG["GROUND_TRUTH_CSV"])
ref_map = dict(zip(gt[CONFIG["GT_Q_COL"]].astype(str).str.strip(), gt[CONFIG["GT_REF_COL"]].astype(str)))
chunks = pd.read_csv(CONFIG["AWMF_CHUNKS_CSV"])[CONFIG["CHUNK_TEXT_COL"]].dropna().astype(str).tolist()
print("AWMF chunks:", len(chunks))

def build_index(m):
    emb = m.encode(chunks, batch_size=64, normalize_embeddings=True, show_progress_bar=True).astype("float32")
    ix = faiss.IndexFlatIP(emb.shape[1]); ix.add(emb); return ix

def retrieval_rows(m):
    ix = build_index(m)
    rows = []
    for q in queries_en:                                   # queries_en is defined in Section 5
        ref = ref_map.get(str(q).strip())
        if not ref:
            continue                                       # score only questions that have a reference
        v = m.encode([str(q)], normalize_embeddings=True).astype("float32")
        _, idx = ix.search(v, CONFIG["TOP_K"])
        rows.append({"question": str(q), "contexts": [chunks[i] for i in idx[0]], "ground_truth": ref})
    return rows

from datasets import Dataset
from ragas import evaluate as ragas_evaluate
from ragas.metrics import context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

judge = LangchainLLMWrapper(ChatOpenAI(
    model = CONFIG["JUDGE_MODEL"], temperature = 0,
    base_url = CONFIG["JUDGE_BASE_URL"], api_key = os.environ["OPENROUTER_API_KEY"]))
judge_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name = CONFIG["BASE_MODEL"]))

def run_ragas(rows):
    ds = Dataset.from_dict({
        "question":     [r["question"]     for r in rows],
        "contexts":     [r["contexts"]     for r in rows],
        "ground_truth": [r["ground_truth"] for r in rows],
    })
    # Recent ragas versions rename these columns to user_input / retrieved_contexts / reference.
    return ragas_evaluate(ds, metrics=[context_precision, context_recall], llm=judge, embeddings=judge_emb)

print("base bge-m3 retrieval RAGAS:")
print(run_ragas(retrieval_rows(base_model)))
print("fine-tuned bge-m3 retrieval RAGAS:")
print(run_ragas(retrieval_rows(tuned_model)))

## 7. Interpretation

A genuine improvement appears as higher P@1, R@5, and MRR for the fine-tuned row than the base row
on the unseen evaluation slice. Because the candidates are the parallel German questions rather
than the AWMF chunks, these values are an alignment proxy rather than the final retrieval score.

The definitive test places the fine-tuned embedder in the retrieval stage of the full pipeline and
re-runs the corpus-grounded RAGAS evaluation of Section 4.6 and Section 4.22, re-embedding the
AWMF chunks with the new model first. In this codebase that is a single change: the retrieval
module reads its query embedder from the BGE_EMBED_URL service seam, so the fine-tuned model can
be served there and measured against the base bge-m3 under the same conditions.

As with the generator experiment, the training set is small and the signal is parallel questions
rather than query-to-chunk relevance, so an alignment gain does not automatically transfer to a
corpus-retrieval gain. The notebook reports the measured quantities without extrapolation.